In [0]:
# Cache() and Persist()

In [0]:
df.cache() # transformation
df.first() / df.count()

df.cache().first()

#### Bucketing

In [0]:
from pyspark.sql import functions as F

df = spark.range(1, 1_000_000+1).withColumn("value", (F.rand() * 100).cast("int"))
df = df.withColumn("year", ((F.rand() * 7).cast("int") + 2020))
display(df)

In [0]:
df.write.parquet("/Volumes/quant_databricks/batch0506/quantcloudrawdatasets/sample_table")

In [0]:
df.write.format("parquet").partitionBy("year").save("/Volumes/quant_databricks/batch0506/quantcloudrawdatasets/partitioned_table")

In [0]:
df.write.format("delta").saveAsTable("quant_databricks.`quant-learning`.non_bucketed_table1")
df.write.format("delta").saveAsTable("quant_databricks.`quant-learning`.non_bucketed_table2")

In [0]:
part_df = spark.read.parquet("/Volumes/quant_databricks/batch0506/quantcloudrawdatasets/partitioned_table/year=2020/")
part_df.display()

In [0]:
display(df.filter(F.col("id") == 124989))

In [0]:
part_df = spark.read.parquet("/Volumes/quant_databricks/batch0506/quantcloudrawdatasets/partitioned_table/year=2022/")

part_df.filter(F.col("id") == 124989).display()

In [0]:
%sql
select 
  *
from `quant_databricks`.`quant-learning`.`non_bucketed_table1` tb1
inner join `quant_databricks`.`quant-learning`.`non_bucketed_table2` tb2
on tb1.id = tb2.id  

In [0]:
df.write.bucketBy(10, "id").sortBy("id").format("parquet").saveAsTable("samples.bakehouse.bucketed_table1")
df.write.bucketBy(10, "id").sortBy("id").format("parquet").saveAsTable("samples.bakehouse.bucketed_table2")

In [0]:
10 diff hash_ids -> each hash id represents 1 bucket

parquet -> start, commit, part-00, part - 01

partition By | parquet -> start, commit, 
2020 -> part-01, part-01 ...,
2021 -> part -00, part - 01...
2022 -> part -00, part - 01...


-- let's say you want to filter a particular order_id which happened in 2022 year

df = df.filter(F.col("order_id") == "1234")

predicate push down 
-- with partition by year
df = df.filter(F.col("year") == 2022 & F.col("order_id") == "1234")



In [0]:
bucket-00/ 2020/2021/2022
2020 -> part-01, part-01 ...,
2021 -> part -00, part - 01...
2022 -> part -00, part - 01...

bucket-01/ 2020/2021/2022
2020 -> part-01, part-01 ...,
2021 -> part -00, part - 01...
2022 -> part -00, part - 01...

bucket-02/ 2020/2021/2022
2020 -> part-01, part-01 ...,
2021 -> part -00, part - 01...
2022 -> part -00, part - 01...

bucket-03/ 2020/2021/2022
2020 -> part-01, part-01 ...,
2021 -> part -00, part - 01...
2022 -> part -00, part - 01...

In [0]:
new_df = spark.read.parquet("/Volumes/quant_databricks/batch0506/quantcloudrawdatasets/partitioned_table/")
new_df.display()